# 01 Data Inspection (Stage 2)

Goal: understand the structure of `cell2celltrain.csv` before any cleaning or modelling.

This notebook only **looks** at the data. It does not change it. The raw file in `data/raw/` is read-only.

## 1. Setup

In [ ]:
# Tools: pandas handles tables of data.
import pandas as pd

# Our own package (src/churn/): shared settings and the shared data loader.
# Using these means paths and column names are written in ONE place only.
from churn.config import ID_COLUMN, RAW_DATA_PATH, TARGET_COLUMN
from churn.data import load_raw_data

# Named constants instead of "magic numbers" scattered in the code.
SAMPLE_ROWS = 5  # rows to show as a sample
TOP_CATEGORIES = 5  # most common values to show per text column
BYTES_PER_MB = 1024**2  # 1,048,576 bytes = 1 MB

# Show all 58 columns instead of hiding the middle ones with "...".
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. File check

Size and line count of the raw file, so we can compare with what pandas loads.

In [ ]:
# File size in MB: Windows gives the size in bytes, we divide to get MB.
file_size_mb = RAW_DATA_PATH.stat().st_size / BYTES_PER_MB

# Count lines one at a time (memory-friendly, works even for huge files).
# "with" closes the file automatically when done.
with RAW_DATA_PATH.open(encoding="utf-8") as raw_file:
    line_count = sum(1 for _ in raw_file)

# :.1f = 1 decimal place, :, = thousands separator (51,048).
print(f"File size: {file_size_mb:.1f} MB")
print(f"Lines in file (including header): {line_count:,}")

## 3. Load the data

If rows loaded is not equal to lines minus the header, some rows are broken or split across lines.

In [ ]:
# Load with our shared loader. It reads the CSV (read only) and also checks:
# required columns exist, no repeated IDs, Churn is only "Yes"/"No".
df = load_raw_data()

# Sanity check: rows loaded should equal lines in the file minus the header.
# If not, some rows are broken or were skipped.
expected_rows = line_count - 1
print(f"Shape (rows, columns): {df.shape}")
print(f"Rows expected from line count: {expected_rows:,}")
print(f"Rows loaded match line count: {len(df) == expected_rows}")

## 4. Column summary

Type, number of distinct values and missing values for every column.

In [ ]:
# How many columns of each type (float = decimals, int = whole numbers, str = text).
print("Columns per type:")
print(df.dtypes.astype(str).value_counts().to_string())

# One table with 4 facts per column:
# - dtype: column type
# - n_unique: number of different values (NaN not counted)
# - n_missing: empty cells (isna gives True/False, sum counts the Trues)
# - pct_missing: share of empty cells as a % (mean of True/False = share True)
column_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean().mul(100).round(2),
    }
)
# The last line of a cell is shown automatically as a table.
column_summary

## 5. Sample rows

`CustomerID` is left out on purpose: we do not print identifiers.

In [ ]:
# Show the first 5 rows WITHOUT CustomerID (we never print identifiers).
# drop() makes a temporary copy for display: df itself is not changed.
df.drop(columns=ID_COLUMN).head(SAMPLE_ROWS)

## 6. Target: `Churn`

In [ ]:
# Count Yes/No once. dropna=False also counts blanks, so missing labels show up.
target_counts = df[TARGET_COLUMN].value_counts(dropna=False)

# Table with counts and % side by side (count / total rows * 100).
target_summary = pd.DataFrame(
    {"count": target_counts, "pct": target_counts.div(len(df)).mul(100).round(2)}
)
target_summary

## 7. Duplicates

- Identical rows (all columns the same)
- Identical rows apart from `CustomerID` (same customer data under two IDs?)
- Repeated `CustomerID` values (is each row really one customer?)

In [ ]:
# 3 duplicate checks. duplicated() marks repeats as True, sum() counts them.
# 1. Exact copies: every column the same.
full_row_duplicates = df.duplicated().sum()
# 2. Same data under a different ID (ignore the ID column, then compare).
duplicates_ignoring_id = df.drop(columns=ID_COLUMN).duplicated().sum()
# 3. Same ID used more than once.
repeated_ids = df[ID_COLUMN].duplicated().sum()

print(f"Identical rows: {full_row_duplicates:,}")
print(f"Identical rows ignoring {ID_COLUMN}: {duplicates_ignoring_id:,}")
print(f"Repeated {ID_COLUMN} values: {repeated_ids:,}")

## 8. Text columns

Number of categories and the most common values, to spot odd labels (for example `Unknown` or typos).

In [ ]:
# List every column that is NOT a number (skipping the ID).
# "Not a number" works on both pandas 2 (object) and pandas 3 (str).
text_columns = [
    column
    for column in df.columns
    if column != ID_COLUMN and not pd.api.types.is_numeric_dtype(df[column])
]

# For each text column: number of categories and the 5 most common values.
# This finds hidden missing values like "Unknown" that isna() cannot see.
print(f"Text columns: {len(text_columns)}\n")
for column in text_columns:
    top_values = df[column].value_counts(dropna=False).head(TOP_CATEGORIES).to_dict()
    print(f"{column} ({df[column].nunique()} categories): {top_values}")

## 9. Number ranges

Look for impossible values, for example negative minutes or charges.

In [ ]:
# Summary stats (count, mean, min, quartiles, max) for every number column.
# The ID is dropped: an "average ID" means nothing.
# .T flips rows and columns, so each column is one row (easier to read).
df.drop(columns=ID_COLUMN).describe().T